In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client= OpenAI()

In [3]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [5]:
from minsearch import Index
index=Index(
    text_fields=['question','section','answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [74]:
question=""

#index.search(
#    question,
#    filter_dict={'course':'llm-zoomcamp'},
#    num_results=2
#    )


In [6]:
question = "Which models can I use?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=1
)

search_results

[{'id': '0d74a3616f',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Agentic RAG',
  'question': 'Any free models with tool use support?',
  'answer': "Several Groq models offer tool use, such as Deepseek R1 or Llama 4, all of which can be used for free for development.\n\nOther providers also support tool or function calling, including Mistral, Gemini, and some local Ollama models.\n\nYou'll typically need to adapt the code when not using OpenAI, because tool schemas and response shapes differ between providers.\n\nFor more details, see the [Groq Tool Use Documentation](https://console.groq.com/docs/tool-use)."}]

In [7]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [8]:
search_results = search("Certificate")

search_results

[{'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '74eb249bbf',
  '

In [9]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [ ]:
USER_PROMPT_TEMPLATE='''
Question: {question}

Context:
{context}
'''.strip()

In [11]:
def build_context(search_results):
    lines=[]

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: "+doc["question"])
        lines.append("A: "+doc["answer"])
        lines.append("")
    
    return "\n".join(lines).strip()

In [12]:
def build_prompt(question,search_results):
    context=build_context(search_results)
    prompt= USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [13]:
prompt=build_prompt(question,search_results)

In [14]:
print(prompt)

Question:
Which models can I use?

Context:
General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project whil

In [15]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text


In [16]:
def rag(query, model="gpt-5.4-mini"):
    search_results=search(query)
    prompt=build_prompt(query, search_results)
    answer=llm(INSTRUCTIONS,prompt,model=model)
    return answer

In [66]:
question="Does the course provide a certificate?"

In [17]:
#question=""
test1=rag("Can I still join the course?")
test2=rag("Do the course provide a certificate?")
print(test1)
print(test2)

Yes, you can still join the course. If you want to receive a certificate, make sure to submit your project while submissions are still open.
Yes, but only if you finish the course with a live cohort. The course does not award certificates for the self-paced mode.
